# 怎样自动决定拿钥匙、开门与送货的顺序？：参考答案

每道题先写预期结果再运行。练习中的 TODO 需要自行完成；能顺序运行不代表已完成练习。

In [1]:
from pathlib import Path
CASE_DIR = Path.cwd()
# 载入本例的可修改函数和数据，所有算法都在同目录 solution.py。
exec(compile((CASE_DIR / "solution.py").read_text(encoding="utf-8"), str(CASE_DIR / "solution.py"), "exec"))

初始事实 ['at-hall', 'key-hall', 'package-hall']
动作计划 ['take-key', 'open-door', 'take-package', 'enter-room', 'deliver']
{
  "plan": [
    "take-key",
    "open-door",
    "take-package",
    "enter-room",
    "deliver"
  ],
  "action_count": 5,
  "visited_states": 9,
  "business_validator_pass": true,
  "draft_plan": [
    "open-door",
    "take-package",
    "enter-room",
    "deliver"
  ],
  "draft_validator_pass": false,
  "external_pddl_solver_run": false
}
采用5步计划：take-key → open-door → take-package → enter-room → deliver。独立业务执行检查通过。漏写钥匙前提的草稿产生4步计划，但独立检查拒绝；求解器只能解已写出的模型，不能自动补齐现实需求。


## 1. 补全：动作前提
初始状态不能执行开门；先拿钥匙后可以。

In [2]:
try:apply_action(initial_state,actions[1])
except ValueError:pass
else:raise AssertionError("无钥匙不应开门")
s=apply_action(initial_state,actions[0]);s=apply_action(s,actions[1])
assert "door-open" in s


## 2. 修改：包裹不存在
从初始状态移除package-hall，原动作模型应报告目标不可达。

In [3]:
changed=set(initial_state)-{"package-hall"}
p,_=plan_actions(changed,target,actions)
assert p is None
print("当前模型内完整状态搜索后未发现可达目标。")


当前模型内完整状态搜索后未发现可达目标。


## 3. 迁移：实验前置步骤
使用同一搜索器表达“未准备→已校准→已实验→报告完成”，并检查缺少校准动作时不能完成。

In [4]:
lab=[action("calibrate",["ready"],["calibrated"],[]),
     action("experiment",["calibrated"],["measured"],[]),
     action("report",["measured"],["reported"],[])]
p,_=plan_actions({"ready"},{"reported"},lab)
assert p==["calibrate","experiment","report"]
assert plan_actions({"ready"},{"reported"},lab[1:])[0] is None
print(p)


['calibrate', 'experiment', 'report']
